In [ ]:
import subprocess
subprocess.run(['pip', 'install', 'torch', 'torchvision'], capture_output=True)

# Direct training without file
import torch
import torch.nn as nn
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader
import torch.nn.functional as F
import math

print("Libraries ready! ")

Libraries ready! 


In [ ]:
import torch, torch.nn as nn, torch.nn.functional as F
import math, os
from typing import List, Tuple, Dict

class PrunableLinear(nn.Module):
    def __init__(self, in_features, out_features):
        super().__init__()
        self.weight = nn.Parameter(torch.empty(out_features, in_features))
        self.bias   = nn.Parameter(torch.zeros(out_features))
        # Start near 0, lambda will push negative = prune
        #self.gate_scores = nn.Parameter(torch.randn(out_features, in_features) * 0.01)
        self.gate_scores = nn.Parameter(torch.zeros(out_features, in_features))
        nn.init.kaiming_uniform_(self.weight, a=math.sqrt(5))

    def forward(self, x):
        gates = torch.sigmoid(self.gate_scores)
        return F.linear(x, self.weight * gates, self.bias)

    def prunable_layers(self): return self

class SelfPruningNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.l1 = PrunableLinear(3072, 512)
        self.l2 = PrunableLinear(512, 256)
        self.l3 = PrunableLinear(256, 10)
        self.bn1 = nn.BatchNorm1d(512)
        self.bn2 = nn.BatchNorm1d(256)

    def forward(self, x):
        x = x.view(x.size(0), -1)
        x = F.relu(self.bn1(self.l1(x)))
        x = F.relu(self.bn2(self.l2(x)))
        return self.l3(x)

    def prunable_layers(self):
        return [self.l1, self.l2, self.l3]

print("Model ready ")

Model ready 


In [ ]:
import torchvision, torchvision.transforms as transforms
from torch.utils.data import DataLoader
import torch.optim as optim
import numpy as np

def sparsity_loss(model):
    total = torch.tensor(0.0, device=next(model.parameters()).device)
    for layer in model.prunable_layers():
        #total = total + torch.sigmoid(layer.gate_scores).sum()
        total = total + torch.sigmoid(layer.gate_scores).abs().sum()
    return total

def get_loaders():
    mean, std = (0.4914,0.4822,0.4465), (0.2470,0.2435,0.2616)
    train_tf = transforms.Compose([
        transforms.RandomHorizontalFlip(),
        transforms.RandomCrop(32, padding=4),
        transforms.ToTensor(),
        transforms.Normalize(mean, std)
    ])
    test_tf = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize(mean, std)
    ])
    tr = torchvision.datasets.CIFAR10('./data', train=True,  download=True, transform=train_tf)
    te = torchvision.datasets.CIFAR10('./data', train=False, download=True, transform=test_tf)
    return DataLoader(tr, 128, shuffle=True, num_workers=2), DataLoader(te, 256, num_workers=2)

def run(lam, epochs, train_loader, test_loader, device):
    model = SelfPruningNet().to(device)
    opt   = optim.Adam(model.parameters(), lr=1e-3)
    sch   = optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs)

    for epoch in range(1, epochs+1):
        model.train()
        for imgs, labels in train_loader:
            imgs, labels = imgs.to(device), labels.to(device)
            opt.zero_grad()
            logits  = model(imgs)
            cls     = F.cross_entropy(logits, labels)
            spar    = sparsity_loss(model)
            loss    = cls + lam * spar
            loss.backward()
            opt.step()
        sch.step()

        if epoch % 5 == 0 or epoch == 1:
            model.eval()
            correct = total = 0
            with torch.no_grad():
                for imgs, labels in test_loader:
                    imgs, labels = imgs.to(device), labels.to(device)
                    correct += (model(imgs).argmax(1) == labels).sum().item()
                    total   += labels.size(0)
            print(f"  λ={lam} | Epoch {epoch}/{epochs} | acc={correct/total*100:.2f}%")

    # Final evaluation
    model.eval()
    correct = total = 0
    all_gates = []
    with torch.no_grad():
        for imgs, labels in test_loader:
            imgs, labels = imgs.to(device), labels.to(device)
            correct += (model(imgs).argmax(1) == labels).sum().item()
            total   += labels.size(0)
        for layer in model.prunable_layers():
            all_gates.append(torch.sigmoid(layer.gate_scores).cpu().flatten())

    gates      = torch.cat(all_gates).numpy()
    #sparsity   = (gates < 0.01).mean() * 100
    sparsity = (gates < 0.2).mean() * 100
    acc        = correct / total * 100
    print(f"\n  -- λ={lam} | Final Acc={acc:.2f}% | Sparsity={sparsity:.2f}%")
    return acc, sparsity, gates

print("Training functions ready ")

Training functions ready 


In [ ]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")

train_loader, test_loader = get_loaders()

# Teen lambda values
lambdas = [1e-5, 1e-3, 1e-1]
results = []

for lam in lambdas:
    print(f"\n{'='*50}")
    print(f"Training with λ = {lam}")
    print(f"{'='*50}")
    acc, sparsity, gates = run(lam, 30, train_loader, test_loader, device)
    results.append((lam, acc, sparsity, gates))

# Summary Table
print(f"\n{'='*50}")
print(f"{'Lambda':>10} {'Accuracy':>12} {'Sparsity':>12}")
print(f"{'-'*36}")
for lam, acc, sp, _ in results:
    print(f"{lam:>10.5f} {acc:>11.2f}% {sp:>11.2f}%")

# Plot gate distributions
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, (lam, acc, sp, gates) in zip(axes, results):
    ax.hist(gates, bins=80, color='steelblue', edgecolor='white')
    ax.set_title(f'λ={lam}\nacc={acc:.1f}% | sparsity={sp:.1f}%')
    ax.set_xlabel('Gate value')
    ax.set_xlim(0, 1)

plt.tight_layout()
os.makedirs('outputs', exist_ok=True)
plt.savefig('outputs/gate_distributions.png', dpi=150)
print("\n[saved] outputs/gate_distributions.png")
print("\nDone!")

Device: cpu

Training with λ = 1e-05
  λ=1e-05 | Epoch 1/30 | acc=44.13%
  λ=1e-05 | Epoch 5/30 | acc=52.42%
  λ=1e-05 | Epoch 10/30 | acc=55.65%
  λ=1e-05 | Epoch 15/30 | acc=57.87%
  λ=1e-05 | Epoch 20/30 | acc=59.31%
  λ=1e-05 | Epoch 25/30 | acc=60.17%
  λ=1e-05 | Epoch 30/30 | acc=60.47%

  -- λ=1e-05 | Final Acc=60.47% | Sparsity=87.61%

Training with λ = 0.001
  λ=0.001 | Epoch 1/30 | acc=44.65%
  λ=0.001 | Epoch 5/30 | acc=51.89%
  λ=0.001 | Epoch 10/30 | acc=54.67%
  λ=0.001 | Epoch 15/30 | acc=56.93%
  λ=0.001 | Epoch 20/30 | acc=58.32%
  λ=0.001 | Epoch 25/30 | acc=59.69%
  λ=0.001 | Epoch 30/30 | acc=59.95%

  -- λ=0.001 | Final Acc=59.95% | Sparsity=99.98%

Training with λ = 0.1
  λ=0.1 | Epoch 1/30 | acc=44.46%
  λ=0.1 | Epoch 5/30 | acc=51.26%
  λ=0.1 | Epoch 10/30 | acc=53.66%
  λ=0.1 | Epoch 15/30 | acc=55.83%
  λ=0.1 | Epoch 20/30 | acc=56.94%
  λ=0.1 | Epoch 25/30 | acc=58.09%
  λ=0.1 | Epoch 30/30 | acc=57.96%

  -- λ=0.1 | Final Acc=57.96% | Sparsity=100.00%

    L